# 7. The language models

A recognizer reads each 4-pixel frame of a line from its pixels alone, so on an unsure line it can pick a
look-alike: `rn` for `m`, `l` for `1`, a faint letter, a dropped accent. A set that ships language models can read
those lines again with a model of the language, and the likelier reading wins:

| `lm_mode` | `carbon-ocr` | what reads the lines under the gate |
|---|---|---|
| `"none"` | `--lm-mode none` | nothing more: every line greedy, as by default |
| `"ngram"` | `--lm-mode ngram` | a CTC prefix beam of 8, scored with the set's character 7-gram (`lm-latin.bin`) |
| `"transformer"` | `--lm-mode transformer` | the beam's best readings rescored by the set's small transformer (`tlm-latin.bin`), each after the line above it |
| `"both"` | `--lm-mode both` | the beam with the 7-gram, its best readings rescored by the transformer |

Each model runs at the weights and gate tuned into its file for the set's recognizer, on held-out lines. A mode
whose model the set does not have is an error. The notebook needs a set that ships them, such as `fullerene-1.1-3M`:
set `CARBON_OCR_SET` to its folder. It reads the sample pages, clean and made harder, in each mode: what it changes,
what it costs, and how to set it by hand.

In [1]:
import time

import numpy as np
from PIL import Image, ImageFilter

import carbon_ocr
import helpers
import sample_pages

SET = helpers.model_set()  # a set with its language models: $CARBON_OCR_SET (fullerene-1.1-3M)
if not (SET / "lm-latin.bin").is_file():
    raise SystemExit(f"{SET} ships no language models: point CARBON_OCR_SET at a set that does (fullerene-1.1-3M)")
pages = sample_pages.make_all("data")
MODES = ["none", "ngram", "transformer", "both"]
try:  # the transformer on the GPU where the module has CUDA and the machine one, else on the CPU
    carbon_ocr.cuda_transformer_lm(str(SET / "tlm-latin.bin"), 0)
    GPU = 0
except RuntimeError:
    GPU = -1


def harder(path, radius=1.8, sigma=32, seed=0):
    """A page blurred and noised, as a poor scan reads: its text the page's own."""
    img = Image.open(path).filter(ImageFilter.GaussianBlur(radius))
    a = np.asarray(img, np.float32) + np.random.default_rng(seed).normal(0, sigma, img.size[::-1])
    return np.clip(a, 0, 255).astype(np.uint8)


def score(page, path):
    """CER line by line against the page's truth (helpers.line_cer), and the page's own CER in reading order."""
    t = sample_pages.truth(path)
    e = carbon_ocr.text_errors([sample_pages.truth_text(path)], [page["text"]]).sum(0)
    return helpers.line_cer(t["lines"], page["lines"]), e[0] / e[1]

## What the set ships

The files' headers hold their tuned weights: the n-gram's `alpha` (its weight), `beta` (a bonus a character, which
offsets the model's cost of each) and `gate` (a line is read again where its confidence is under it; 2 reads every
line); the transformer's weight and gate.

In [2]:
ng = carbon_ocr.NgramModel(str(SET / "lm-latin.bin"))
tf = carbon_ocr.TransformerLM(str(SET / "tlm-latin.bin"))
print("lm-latin.bin  ", {k: ng.header[k] for k in ("order", "alpha", "beta", "gate")})
print("tlm-latin.bin ", {k: tf.header[k] for k in ("precision", "d", "layers", "ctx", "alpha", "gate", "params")})

lm-latin.bin   {'order': 7, 'alpha': 0.30000001192092896, 'beta': 2.0, 'gate': 2.0}
tlm-latin.bin  {'precision': 'fp32', 'd': 128, 'layers': 4, 'ctx': 128, 'alpha': 0.10000000149011612, 'gate': 2.0, 'params': 912256}


## Each mode on clean and harder pages

The same pages read by one engine a mode. On clean print the recognizer is sure of nearly every line, so the models
have little to change; on a poor scan they read many lines again.

In [3]:
names = ["letter", "columns", "table", "receipt"]
images = {n: np.asarray(Image.open(pages[n])) for n in names}
images.update({f"{n} (blurred, noisy)": harder(pages[n]) for n in names})
paths = {**{n: pages[n] for n in names}, **{f"{n} (blurred, noisy)": pages[n] for n in names}}
results = {}
for mode in MODES:
    engine = carbon_ocr.Engine(SET, lm_mode=mode, **({"tlm_gpu": GPU} if mode in ("transformer", "both") else {}))
    engine.ocr([images["letter"]])  # one page first: the models load and warm up
    batch, times = list(images.values()) * 4, []
    for _ in range(3):  # three rounds of the pages four times over; the median round
        t = time.perf_counter()
        read = engine.ocr(batch)
        times.append(time.perf_counter() - t)
    results[mode] = (dict(zip(images, read)), sorted(times)[1] / 4)
    del engine
print(f"{'page':28}" + "".join(f"{m:>13}" for m in MODES))
for n in images:
    print(f"{n:28}" + "".join(f"{100 * score(results[m][0][n], paths[n])[0]:12.2f}%" for m in MODES))
print(f"{'pages a second':28}" + "".join(f"{len(images) / results[m][1]:13.1f}" for m in MODES))

page                                 none        ngram  transformer         both
letter                              0.07%        0.07%        0.00%        0.07%
columns                             1.72%        1.64%        1.72%        1.72%
table                               0.18%        0.00%        0.18%        0.00%
receipt                             0.00%        0.00%        0.00%        0.00%
letter (blurred, noisy)             0.15%        0.15%        0.15%        0.15%
columns (blurred, noisy)            6.09%        5.70%        6.01%        5.78%
table (blurred, noisy)              0.72%        0.18%        0.18%        0.36%
receipt (blurred, noisy)            0.72%        0.97%        0.72%        0.72%
pages a second                       73.6         62.9         58.1         56.0


The numbers are each page's character error rate, line by line against the text it was drawn from. The speeds
are the median of three rounds over the pages, on this machine's CPU (the transformer on its GPU where it has one);
these few small pages flatter every mode, and the benchmarks' pages give the costs in the set's model card.

## What changed

The lines one mode reads differently from greedy, on the harder pages:

In [4]:
for n in [k for k in images if "blurred" in k]:
    base = {tuple(round(v) for v in l["box"]): l["text"] for l in results["none"][0][n]["lines"]}
    for mode in ["ngram", "both"]:
        for l in results[mode][0][n]["lines"]:
            b = tuple(round(v) for v in l["box"])
            if b in base and base[b] != l["text"]:
                print(f"{n:28} {mode:6}  {base[b]!r}\n{'':28} {'':6}  {l['text']!r}")

columns (blurred, noisy)     ngram   'fifty minutes slow each day so high water'
                                     'fifty minutes slow each day, so high water'
columns (blurred, noisy)     ngram   'differ,'
                                     'differ.'
columns (blurred, noisy)     ngram   '* Heights are given agamst chart datum, the level of the'
                                     '* Heights are given against chart datum, the level of the'
columns (blurred, noisy)     ngram   'lowest astroeomicaf tide.'
                                     'lowest astronomical tide.'
columns (blurred, noisy)     both    'differ,'
                                     'differ.'
columns (blurred, noisy)     both    '* Heights are given agamst chart datum, the level of the'
                                     '* Heights are given against chart datum, the level of the'
columns (blurred, noisy)     both    'lowest astroeomicaf tide.'
                                     'lowest astronomical tide.'
tab

A language model makes the likelier text win, which is not always the text printed: an e-mail address, a code
or an amount is not a word, and a model of words can pull it toward one, as the 7-gram reads the table's `Q4`
as `04` above.
That is why each set's weights and gate were tuned not to read a guard set of codes, amounts, dates and IDs worse
than greedy, and why `none` stays the default.

## Setting the weights by hand

`lm_gate`, `lm_alpha`, `lm_beta` and `tlm_alpha` override the files' weights; `beam_width` the beam's width (8,
the width the models were tuned at). A lower gate reads fewer lines again: faster, and nearer greedy.

In [5]:
page = images["columns (blurred, noisy)"]
for options in [dict(lm_mode="none"), dict(lm_mode="ngram"), dict(lm_mode="ngram", lm_gate=0.9),
                dict(lm_mode="ngram", lm_alpha=0.8), dict(lm_mode="ngram", beam_width=4)]:
    engine = carbon_ocr.Engine(SET, **options)
    engine.ocr([page])
    times = []
    for _ in range(5):  # the median of five reads
        t = time.perf_counter()
        out = engine.ocr([page])[0]
        times.append(time.perf_counter() - t)
    ms = 1000 * sorted(times)[2]
    print(f"{str(options):48} CER {100 * score(out, pages['columns'])[0]:5.2f}%  {ms:6.0f} ms")
    del engine

{'lm_mode': 'none'}                              CER  6.09%      66 ms


{'lm_mode': 'ngram'}                             CER  5.70%      69 ms


{'lm_mode': 'ngram', 'lm_gate': 0.9}             CER  5.93%      65 ms


{'lm_mode': 'ngram', 'lm_alpha': 0.8}            CER  5.85%      77 ms


{'lm_mode': 'ngram', 'beam_width': 4}            CER  5.70%      71 ms


## On the command line

`carbon-ocr` and `carbon-ocr serve` take the same switch: `--lm-mode none|ngram|transformer|both`, and `--lm-gate`,
`--lm-alpha`, `--lm-beta`, `--tlm-alpha`, `--tlm-gpu N` beside it.

In [6]:
import subprocess

exe = helpers.carbon_ocr_program()
if exe:
    for mode in ["none", "ngram"]:
        out = subprocess.run([exe, "ocr", str(pages["receipt"]), "--set", str(SET), "--lm-mode", mode],
                             capture_output=True, text=True)
        print(f"--lm-mode {mode}:", out.stdout.strip().splitlines()[:3])
else:
    print("carbon-ocr not found: build it with CMake (notebook 6)")

--lm-mode none: ['HARBOUR CAFE', 'North Pier', '14/03/2026']


--lm-mode ngram: ['HARBOUR CAFE', 'North Pier', '14/03/2026']


## What to expect

On the benchmarks the set was measured on (each graded by its own grader), the 7-gram gains about a point on
OmniParseBench and on olmOCR-Bench at little cost in speed; the transformer adds a little more on olmOCR-Bench and
costs more time where it rescores every line. Neither writes a character the recognizer did not rank for its frame,
nor reads math, scripts the recognizer does not read, or most handwriting. On codes, amounts and IDs a language model
can pull a reading toward a word: the weights were tuned not to read them worse than greedy.